# MealSense 9-Class VLM Component & Portion Analyzer

Fine-tune and run a lightweight Vision-Language Model (`SmolVLM-500M-Instruct` or `Qwen2-VL-2B-Instruct`) with LoRA to estimate visible ingredients, preparation state, and portion fullness for 9 Bangladeshi dishes, fused with the **FCTB 2013 (INFS, University of Dhaka)** SQLite nutrition database.

## Key Optimizations & Modern LTS Design
1. **GPU-First PyTorch Setup**: Detects NVIDIA GPUs and flags CPU-only PyTorch wheels (`+cpu`) on Windows; enables TF32, `bfloat16`, `adamw_torch_fused`, and PyTorch SDPA attention.
2. **Eliminates Image-Splitting Bottleneck**: Disables `Idefics3` 17-crop 2048px upscaling by default (`do_image_splitting=False`, `longest_edge=384`), cutting vision tokens by >10x for faster training and inference.
3. **Assistant-Only Loss Masking**: Dynamically masks the entire prompt prefix (system prompt, `<image>` tokens, user instructions) up to the assistant turn delimiter, and masks padding via `attention_mask == 0` (safe even when `pad_token_id == eos_token_id`).
4. **Native `bfloat16` LoRA for 500M Models**: Skips slow 4-bit NF4 dequantization and unnecessary gradient checkpointing on 8 GB GPUs when running 500M models, while retaining automatic 4-bit QLoRA fallback for 2B+ models or tight VRAM.
5. **Modern Python 3.12+/3.13 & Transformers 5.x LTS**: Uses `dtype=`, `eval_strategy=`, `TaskType.CAUSAL_LM`, `torch.inference_mode()`, `dataclass(frozen=True, slots=True)`, `sqlite3` context managers with caching, and **Pydantic v2** schema validation.

## 1. Environment & GPU Dependencies

> **Why did the previous notebook run on CPU?**  
> On Windows, `pip install torch torchvision torchaudio` without `--index-url` installs the CPU-only wheel (`+cpu`), even when an NVIDIA GPU (such as an RTX 5050) is present.  
> Run the cell below to inspect your CUDA status and optionally install CUDA-enabled PyTorch (`cu128` or `cu130` for RTX 50-series Blackwell or Ampere/Ada GPUs).

In [1]:
import shutil
import subprocess
import sys


def check_gpu_and_torch_status() -> None:
    has_nvidia_smi = shutil.which("nvidia-smi") is not None
    if has_nvidia_smi:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            check=False,
        )
        print("Detected NVIDIA GPU:", smi.stdout.strip() or "Unknown")
    else:
        print("nvidia-smi not found on PATH.")

    try:
        import torch
        print(f"Installed PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
        if has_nvidia_smi and not torch.cuda.is_available():
            print(
                "\n[ACTION REQUIRED] PyTorch is a CPU build (+cpu) while an NVIDIA GPU is present!\n"
                "Set INSTALL_CUDA_TORCH = True below (requires ~4-6 GB temporary free disk space),\n"
                "or run in terminal:\n"
                "  .\\.venv\\Scripts\\python.exe -m pip install --no-cache-dir --force-reinstall "
                "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128"
            )
    except ImportError:
        print("PyTorch is not installed yet.")


# Toggle these flags to install packages or switch from +cpu PyTorch to CUDA PyTorch
INSTALL_CUDA_TORCH = False
INSTALL_PACKAGES = False
CUDA_WHEEL_INDEX = "https://download.pytorch.org/whl/cu128"  # cu128 or cu130 for RTX 50-series (Blackwell)

if INSTALL_CUDA_TORCH:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
        "torch", "torchvision", "torchaudio", "--index-url", CUDA_WHEEL_INDEX,
    ])

if INSTALL_PACKAGES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "transformers>=5.0.0", "accelerate>=1.5.0", "peft>=0.15.0",
        "bitsandbytes>=0.46.0", "datasets>=3.0.0", "pydantic>=2.10.0",
        "pillow>=11.0.0", "huggingface_hub>=0.30.0", "requests>=2.32.0",
    ])

check_gpu_and_torch_status()

Detected NVIDIA GPU: NVIDIA GeForce RTX 5050, 8151 MiB, 595.76
Installed PyTorch: 2.11.0+cu128 | CUDA available: True


## 2. Configuration, Cache Guard & Hardware Setup

Environment variables (`HF_HOME`, `HF_HUB_CACHE`, `HF_HUB_DISABLE_XET`, `TMP`, `TEMP`) are set **before** importing `huggingface_hub` or `transformers` so caches and temporary files stay on the workspace drive.

In [2]:
from __future__ import annotations

import os
import shutil
from dataclasses import dataclass
from pathlib import Path

WORKSPACE_DIR = Path(".").resolve()
ARTIFACTS_DIR = WORKSPACE_DIR / "mealsense_local_artifacts"
LOCAL_HF_CACHE = ARTIFACTS_DIR / "hf_cache"
LOCAL_TEMP_DIR = ARTIFACTS_DIR / "tmp"
CHECKPOINT_DIR = WORKSPACE_DIR / "ckpoint" / "mealsense-vlm-lora"
OUTPUTS_DIR = WORKSPACE_DIR / "outputs"

for _folder in (ARTIFACTS_DIR, LOCAL_HF_CACHE, LOCAL_TEMP_DIR, CHECKPOINT_DIR, OUTPUTS_DIR):
    _folder.mkdir(parents=True, exist_ok=True)

# Configure caches BEFORE importing huggingface_hub / transformers
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TMP"] = str(LOCAL_TEMP_DIR)
os.environ["TEMP"] = str(LOCAL_TEMP_DIR)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from huggingface_hub import snapshot_download

ALLOWED_FOOD_CLASSES: tuple[str, ...] = (
    "bakorkhani",
    "beguni",
    "biriyani",
    "chickpeas",
    "egg_omlete",
    "fuchka",
    "haleem",
    "hilsha_fish",
    "kacchi",
)


@dataclass(frozen=True, slots=True)
class MealSenseConfig:
    workspace_dir: Path = WORKSPACE_DIR
    food_train_dir: Path = WORKSPACE_DIR / "food_dataset" / "train"
    artifacts_dir: Path = ARTIFACTS_DIR
    checkpoint_dir: Path = CHECKPOINT_DIR
    outputs_dir: Path = OUTPUTS_DIR
    sqlite_db_path: Path = ARTIFACTS_DIR / "fctb_2013_nutrition.db"
    # Model options: "HuggingFaceTB/SmolVLM-500M-Instruct" (~1 GB) or "Qwen/Qwen2-VL-2B-Instruct" (~4.2 GB)
    hf_repo_id: str = "HuggingFaceTB/SmolVLM-500M-Instruct"
    image_size: int = 384
    # Disable 17-crop 2048px upscaling on SmolVLM/Idefics3 for >10x faster training & inference
    enable_image_splitting: bool = False
    max_images_per_class: int = 50
    val_fraction: float = 0.15
    seed: int = 42
    num_train_epochs: int = 3
    learning_rate: float = 2e-4
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    @property
    def local_base_model_dir(self) -> Path:
        return self.artifacts_dir / self.hf_repo_id.split("/")[-1]


@dataclass(frozen=True, slots=True)
class RuntimeSpec:
    device: str
    dtype: torch.dtype
    free_vram_gb: float
    total_vram_gb: float
    use_4bit: bool


def configure_runtime(cfg: MealSenseConfig, require_gpu: bool = False) -> RuntimeSpec:
    free_d_gb = shutil.disk_usage(cfg.workspace_dir).free / (1024 ** 3)
    free_c_gb = shutil.disk_usage(Path.home()).free / (1024 ** 3)
    print(f"Disk Free -> Workspace: {free_d_gb:.2f} GB | Home: {free_c_gb:.2f} GB")

    if torch.cuda.is_available():
        torch.set_float32_matmul_precision("high")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

        free_bytes, total_bytes = torch.cuda.mem_get_info(0)
        free_vram_gb = free_bytes / (1024 ** 3)
        total_vram_gb = total_bytes / (1024 ** 3)
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        is_large_model = any(tag in cfg.hf_repo_id for tag in ("2B", "3B", "7B"))
        use_4bit = is_large_model or free_vram_gb < 4.0

        gpu_name = torch.cuda.get_device_name(0)
        print(
            f"GPU Ready -> {gpu_name} | VRAM: {free_vram_gb:.2f}/{total_vram_gb:.2f} GB free | "
            f"Dtype: {dtype} | 4-bit QLoRA: {use_4bit}"
        )
        return RuntimeSpec("cuda", dtype, free_vram_gb, total_vram_gb, use_4bit)

    if shutil.which("nvidia-smi"):
        msg = (
            f"NVIDIA GPU detected by nvidia-smi, but torch ({torch.__version__}) has CUDA disabled. "
            "Install CUDA PyTorch in Section 1 to run on GPU."
        )
        if require_gpu:
            raise RuntimeError(msg)
        print(f"[WARNING] {msg}")

    print("Running on CPU fallback | Dtype: torch.float32")
    return RuntimeSpec("cpu", torch.float32, 0.0, 0.0, False)


def ensure_local_model(cfg: MealSenseConfig) -> Path:
    model_dir = cfg.local_base_model_dir
    if (model_dir / "config.json").exists():
        print(f"Using cached local model: {model_dir}")
        return model_dir

    print(f"Downloading {cfg.hf_repo_id} to {model_dir}...")
    snapshot_download(
        repo_id=cfg.hf_repo_id,
        local_dir=str(model_dir),
        ignore_patterns=["*.gguf", "*.onnx", "onnx/*"],
        max_workers=2,
    )
    print(f"Model ready at: {model_dir}")
    return model_dir


CFG = MealSenseConfig()
RUNTIME = configure_runtime(CFG, require_gpu=False)
LOCAL_MODEL_DIR = ensure_local_model(CFG)

d:\CS\Hackathon\Mealsense_Models\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Disk Free -> Workspace: 1.06 GB | Home: 2.38 GB
GPU Ready -> NVIDIA GeForce RTX 5050 | VRAM: 6.90/7.96 GB free | Dtype: torch.bfloat16 | 4-bit QLoRA: False
Using cached local model: D:\CS\Hackathon\Mealsense_Models\mealsense_local_artifacts\SmolVLM-500M-Instruct


## 3. Verified Nutrition Database (FCTB 2013 SQLite + Cached API Fallback)

Uses a parameterized SQLite repository (`with sqlite3.connect(...)`) backed by an in-memory lookup cache and a persistent `requests.Session` for optional OpenFoodFacts fallback.

In [3]:
import sqlite3
import requests

FCTB_2013_INGREDIENTS: tuple[tuple[str, str, str, float, float, float, float, float, str], ...] = (
    ("wheat_flour_maida", "Refined Wheat Flour (Maida), raw", "01_0023", 0.90, 348.0, 11.0, 74.1, 0.9, "FCTB 2013 (Code 01_0023) - INFS, DU"),
    ("basmati_rice", "Rice, aromatic/parboiled (Chinigura/Basmati), raw", "01_0012", 2.75, 349.0, 7.5, 77.4, 0.8, "FCTB 2013 (Code 01_0012) - INFS, DU"),
    ("chickpeas_dry", "Chickpea (Chola/Boot), whole dried, raw", "02_0003", 2.40, 347.0, 20.8, 55.0, 5.1, "FCTB 2013 (Code 02_0003) - INFS, DU"),
    ("besan_gram_flour", "Bengal Gram Flour (Besan), raw", "02_0004", 1.00, 372.0, 22.0, 58.2, 5.6, "FCTB 2013 (Code 02_0004) - INFS, DU"),
    ("mixed_lentils", "Mixed Lentils (Masoor, Moong, Mashkalai), raw", "02_0008", 2.50, 336.0, 24.3, 55.8, 1.1, "FCTB 2013 (Group 02 Pulses) - INFS, DU"),
    ("broken_wheat", "Broken Wheat (Gom/Dalia), raw", "01_0021", 2.60, 341.0, 11.8, 71.2, 1.5, "FCTB 2013 (Code 01_0021) - INFS, DU"),
    ("eggplant", "Brinjal / Eggplant (Begun), purple, raw", "03_0007", 0.85, 25.0, 1.4, 4.2, 0.3, "FCTB 2013 (Code 03_0007) - INFS, DU"),
    ("potato", "Potato, local (Gol Alu), raw", "05_0001", 0.92, 66.0, 1.3, 14.8, 0.1, "FCTB 2013 (Code 05_0001) - INFS, DU"),
    ("onion", "Onion (Peyaj), raw", "07_0012", 0.65, 50.0, 1.4, 10.6, 0.1, "FCTB 2013 (Code 07_0012) - INFS, DU"),
    ("green_chili", "Chili, green, raw", "07_0003", 0.95, 41.0, 2.6, 6.1, 0.6, "FCTB 2013 (Code 07_0003) - INFS, DU"),
    ("tamarind_pulp", "Tamarind pulp (Tetul), ripe", "04_0042", 1.00, 283.0, 3.1, 67.4, 0.1, "FCTB 2013 (Code 04_0042) - INFS, DU"),
    ("mustard_seeds", "Mustard seeds (Shorshe), dried", "06_0007", 1.80, 508.0, 20.2, 21.6, 39.7, "FCTB 2013 (Code 06_0007) - INFS, DU"),
    ("mixed_spices", "Bangladeshi Garam Masala & Spices", "07_0005", 1.00, 354.0, 13.5, 44.2, 14.0, "FCTB 2013 (Group 07 Spices) - INFS, DU"),
    ("hilsa_fish", "Hilsa Fish (Ilish), raw edible portion", "09_0028", 0.82, 273.0, 21.8, 0.0, 19.4, "FCTB 2013 (Code 09_0028) - INFS, DU"),
    ("chicken_meat", "Chicken meat, broiler, raw", "10_0008", 0.74, 145.0, 21.8, 0.0, 6.4, "FCTB 2013 (Code 10_0008) - INFS, DU"),
    ("mutton", "Goat meat (Mutton/Khashi), moderate fat, raw", "10_0006", 0.72, 194.0, 21.4, 0.0, 12.1, "FCTB 2013 (Code 10_0006) - INFS, DU"),
    ("beef", "Beef, moderate fat, raw", "10_0002", 0.70, 226.0, 20.6, 0.0, 15.9, "FCTB 2013 (Code 10_0002) - INFS, DU"),
    ("chicken_egg", "Hen Egg (Murgi Dim), whole, raw", "11_0002", 0.92, 158.0, 13.3, 0.6, 11.5, "FCTB 2013 (Code 11_0002) - INFS, DU"),
    ("yogurt", "Curd / Yogurt (Doi), whole milk", "12_0005", 1.00, 97.0, 3.5, 11.4, 4.1, "FCTB 2013 (Code 12_0005) - INFS, DU"),
    ("ghee", "Ghee / Clarified Butter", "13_0002", 1.00, 898.0, 0.0, 0.0, 99.8, "FCTB 2013 (Code 13_0002) - INFS, DU"),
    ("soybean_oil", "Soybean cooking oil", "13_0008", 1.00, 900.0, 0.0, 0.0, 100.0, "FCTB 2013 (Code 13_0008) - INFS, DU"),
    ("mustard_oil", "Mustard oil (Khati Shorsher Tel)", "13_0005", 1.00, 900.0, 0.0, 0.0, 100.0, "FCTB 2013 (Code 13_0005) - INFS, DU"),
    ("sugar", "Sugar, white refined", "14_0003", 1.00, 398.0, 0.0, 99.4, 0.0, "FCTB 2013 (Code 14_0003) - INFS, DU"),
)


@dataclass(frozen=True, slots=True)
class NutritionRecord:
    ingredient_key: str
    display_name: str
    fctb_code: str
    cooking_yield_factor: float
    calories_kcal_100g: float
    protein_g_100g: float
    carbohydrates_g_100g: float
    fat_g_100g: float
    source_citation: str


class NutritionDB:
    """In-memory-cached repository backed by the FCTB 2013 SQLite database."""

    def __init__(self, db_path: Path) -> None:
        self.db_path = db_path
        self._cache: dict[str, NutritionRecord | None] = {}
        self._session = requests.Session()
        self._init_db()

    def _init_db(self) -> None:
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('''
                CREATE TABLE IF NOT EXISTS nutrition_fctb (
                    ingredient_key TEXT PRIMARY KEY,
                    display_name TEXT NOT NULL,
                    fctb_code TEXT NOT NULL,
                    cooking_yield_factor REAL NOT NULL,
                    calories_kcal_100g REAL NOT NULL,
                    protein_g_100g REAL NOT NULL,
                    carbohydrates_g_100g REAL NOT NULL,
                    fat_g_100g REAL NOT NULL,
                    source_citation TEXT NOT NULL
                )
            ''')
            conn.executemany(
                "INSERT OR REPLACE INTO nutrition_fctb VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                FCTB_2013_INGREDIENTS,
            )
            conn.commit()
        print(f"FCTB 2013 SQLite DB ready at {self.db_path} ({len(FCTB_2013_INGREDIENTS)} ingredients).")

    def lookup(self, ingredient_key: str, allow_remote_fallback: bool = True) -> NutritionRecord | None:
        if ingredient_key in self._cache:
            return self._cache[ingredient_key]

        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            row = conn.execute(
                "SELECT * FROM nutrition_fctb WHERE ingredient_key = ?",
                (ingredient_key,),
            ).fetchone()

        if row is not None:
            record = NutritionRecord(**dict(row))
            self._cache[ingredient_key] = record
            return record

        if allow_remote_fallback:
            record = self._fetch_openfoodfacts(ingredient_key)
            self._cache[ingredient_key] = record
            return record

        return None

    def _fetch_openfoodfacts(self, ingredient_key: str) -> NutritionRecord | None:
        url = "https://world.openfoodfacts.org/cgi/search.pl"
        params = {
            "search_terms": ingredient_key,
            "search_simple": 1,
            "action": "process",
            "json": 1,
            "page_size": 1,
        }
        try:
            resp = self._session.get(url, params=params, timeout=5)
            resp.raise_for_status()
            products = resp.json().get("products", [])
            if products and "nutriments" in products[0]:
                nutr = products[0]["nutriments"]
                return NutritionRecord(
                    ingredient_key=ingredient_key,
                    display_name=products[0].get("product_name", ingredient_key),
                    fctb_code="OPENFOODFACTS",
                    cooking_yield_factor=1.0,
                    calories_kcal_100g=float(nutr.get("energy-kcal_100g", 0.0)),
                    protein_g_100g=float(nutr.get("proteins_100g", 0.0)),
                    carbohydrates_g_100g=float(nutr.get("carbohydrates_100g", 0.0)),
                    fat_g_100g=float(nutr.get("fat_100g", 0.0)),
                    source_citation="OpenFoodFacts Public Database (Keyless REST API)",
                )
        except Exception:
            pass
        return None


NUTRITION_DB = NutritionDB(CFG.sqlite_db_path)

FCTB 2013 SQLite DB ready at D:\CS\Hackathon\Mealsense_Models\mealsense_local_artifacts\fctb_2013_nutrition.db (23 ingredients).


## 4. Culinary Knowledge Base & Pydantic v2 Structured Schemas

Defines the 9 Bangladeshi dish priors (from FAO/INFOODS FCT) and strict **Pydantic v2** models for VLM output validation and final nutrition prediction.

In [4]:
from typing import Any, Literal
from pydantic import BaseModel, Field

FAO_RECIPE_SOURCE = (
    "https://www.fao.org/fileadmin/templates/food_composition/documents/FCT_10_2_14_final_version.pdf"
)


class VisibleIngredient(BaseModel):
    name: str
    visual_confidence: float = Field(ge=0.0, le=1.0)
    estimated_volume_share: float = Field(ge=0.0, le=1.0)


class VLMVisionOutput(BaseModel):
    dish_matches_prediction: bool = True
    visual_fullness_multiplier: float = Field(default=1.0, ge=0.25, le=3.0)
    portion_confidence: Literal["high", "medium", "low"] = "medium"
    visible_ingredients: list[VisibleIngredient] = Field(default_factory=list)
    observed_preparation_state: str = ""
    occluded_or_uncertain: bool = False


class ServingEstimate(BaseModel):
    estimated_weight_g: float
    weight_range_g: tuple[float, float]
    confidence: Literal["high", "medium", "low"]


class IngredientBreakdown(BaseModel):
    name: str
    weight_g: float
    identification: Literal["visible", "recipe_inferred"]
    confidence: float
    nutrition_source: str
    recipe_source: str


class MacroSummary(BaseModel):
    calories_kcal: float
    protein_g: float
    carbohydrates_g: float
    fat_g: float


class MealSensePrediction(BaseModel):
    food_class: str
    food_name: str
    recognition_confidence: float
    observed_preparation_state: str
    occluded_or_uncertain: bool
    serving: ServingEstimate
    ingredients: list[IngredientBreakdown]
    nutrition: MacroSummary
    sources: list[str]
    requires_user_confirmation: bool


NINE_CLASS_KNOWLEDGE_BASE: dict[str, dict[str, Any]] = {
    "bakorkhani": {
        "food_name": "Old Dhaka Bakorkhani",
        "standard_serving_g": 80.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Layered unleavened flatbread baked in a clay tandoor with maida, ghee/oil, and sugar.",
        "ingredients": [
            {"name": "wheat_flour_maida", "ratio": 0.72, "identification": "visible", "conf": 0.96},
            {"name": "ghee", "ratio": 0.18, "identification": "recipe_inferred", "conf": 0.88},
            {"name": "soybean_oil", "ratio": 0.06, "identification": "recipe_inferred", "conf": 0.85},
            {"name": "sugar", "ratio": 0.04, "identification": "recipe_inferred", "conf": 0.85},
        ],
    },
    "beguni": {
        "food_name": "Beguni (Battered Fried Eggplant)",
        "standard_serving_g": 100.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Thinly sliced purple eggplant dipped in spiced Bengal gram flour (besan) batter and deep-fried.",
        "ingredients": [
            {"name": "eggplant", "ratio": 0.52, "identification": "visible", "conf": 0.94},
            {"name": "besan_gram_flour", "ratio": 0.30, "identification": "visible", "conf": 0.95},
            {"name": "soybean_oil", "ratio": 0.15, "identification": "recipe_inferred", "conf": 0.88},
            {"name": "mixed_spices", "ratio": 0.03, "identification": "recipe_inferred", "conf": 0.82},
        ],
    },
    "biriyani": {
        "food_name": "Bangladeshi Chicken/Beef Biriyani",
        "standard_serving_g": 350.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Aromatic Chinigura/Basmati rice cooked with spiced chicken or beef, potatoes, onion beresta, and ghee.",
        "ingredients": [
            {"name": "basmati_rice", "ratio": 0.52, "identification": "visible", "conf": 0.96},
            {"name": "chicken_meat", "ratio": 0.28, "identification": "visible", "conf": 0.92},
            {"name": "potato", "ratio": 0.09, "identification": "visible", "conf": 0.88},
            {"name": "onion", "ratio": 0.04, "identification": "recipe_inferred", "conf": 0.84},
            {"name": "ghee", "ratio": 0.04, "identification": "recipe_inferred", "conf": 0.84},
            {"name": "yogurt", "ratio": 0.02, "identification": "recipe_inferred", "conf": 0.80},
            {"name": "mixed_spices", "ratio": 0.01, "identification": "recipe_inferred", "conf": 0.80},
        ],
    },
    "chickpeas": {
        "food_name": "Chola Bhuna (Spiced Chickpeas)",
        "standard_serving_g": 160.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Boiled brown chickpeas sauteed with diced potatoes, chopped onions, green chilies, and roasted spices.",
        "ingredients": [
            {"name": "chickpeas_dry", "ratio": 0.68, "identification": "visible", "conf": 0.96},
            {"name": "potato", "ratio": 0.14, "identification": "visible", "conf": 0.89},
            {"name": "onion", "ratio": 0.08, "identification": "visible", "conf": 0.90},
            {"name": "green_chili", "ratio": 0.02, "identification": "visible", "conf": 0.88},
            {"name": "soybean_oil", "ratio": 0.06, "identification": "recipe_inferred", "conf": 0.85},
            {"name": "mixed_spices", "ratio": 0.02, "identification": "recipe_inferred", "conf": 0.82},
        ],
    },
    "egg_omlete": {
        "food_name": "Bangladeshi Dim Bhaji (Spiced Egg Omelette)",
        "standard_serving_g": 75.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Beaten hen eggs pan-fried in oil with finely sliced onions and green chilies.",
        "ingredients": [
            {"name": "chicken_egg", "ratio": 0.76, "identification": "visible", "conf": 0.97},
            {"name": "onion", "ratio": 0.12, "identification": "visible", "conf": 0.91},
            {"name": "green_chili", "ratio": 0.03, "identification": "visible", "conf": 0.89},
            {"name": "soybean_oil", "ratio": 0.09, "identification": "recipe_inferred", "conf": 0.86},
        ],
    },
    "fuchka": {
        "food_name": "Bangladeshi Fuchka with Tetul Tok",
        "standard_serving_g": 180.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Crisp fried wheat shells stuffed with mashed spiced potatoes, chickpeas, grated egg, onion, and sour tamarind water.",
        "ingredients": [
            {"name": "potato", "ratio": 0.36, "identification": "visible", "conf": 0.92},
            {"name": "chickpeas_dry", "ratio": 0.20, "identification": "visible", "conf": 0.90},
            {"name": "wheat_flour_maida", "ratio": 0.18, "identification": "visible", "conf": 0.95},
            {"name": "tamarind_pulp", "ratio": 0.10, "identification": "visible", "conf": 0.88},
            {"name": "chicken_egg", "ratio": 0.06, "identification": "visible", "conf": 0.87},
            {"name": "onion", "ratio": 0.04, "identification": "visible", "conf": 0.88},
            {"name": "soybean_oil", "ratio": 0.06, "identification": "recipe_inferred", "conf": 0.84},
        ],
    },
    "haleem": {
        "food_name": "Bangladeshi Shahi Mutton/Beef Haleem",
        "standard_serving_g": 280.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Slow-cooked savory porridge of mixed lentils, broken wheat, and tender beef/mutton garnished with fried onions and green chilies.",
        "ingredients": [
            {"name": "beef", "ratio": 0.34, "identification": "visible", "conf": 0.91},
            {"name": "mixed_lentils", "ratio": 0.32, "identification": "visible", "conf": 0.93},
            {"name": "broken_wheat", "ratio": 0.18, "identification": "recipe_inferred", "conf": 0.86},
            {"name": "onion", "ratio": 0.06, "identification": "visible", "conf": 0.90},
            {"name": "green_chili", "ratio": 0.02, "identification": "visible", "conf": 0.88},
            {"name": "soybean_oil", "ratio": 0.06, "identification": "recipe_inferred", "conf": 0.84},
            {"name": "mixed_spices", "ratio": 0.02, "identification": "recipe_inferred", "conf": 0.84},
        ],
    },
    "hilsha_fish": {
        "food_name": "Shorshe Ilish / Fried Hilsha Fish",
        "standard_serving_g": 180.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Hilsa fish steak cooked in pungent mustard seed paste, mustard oil, and whole green chilies.",
        "ingredients": [
            {"name": "hilsa_fish", "ratio": 0.68, "identification": "visible", "conf": 0.97},
            {"name": "mustard_seeds", "ratio": 0.12, "identification": "visible", "conf": 0.89},
            {"name": "green_chili", "ratio": 0.04, "identification": "visible", "conf": 0.91},
            {"name": "mustard_oil", "ratio": 0.10, "identification": "recipe_inferred", "conf": 0.88},
            {"name": "onion", "ratio": 0.06, "identification": "recipe_inferred", "conf": 0.82},
        ],
    },
    "kacchi": {
        "food_name": "Old Dhaka Mutton Kacchi Biryani",
        "standard_serving_g": 380.0,
        "recipe_source": FAO_RECIPE_SOURCE,
        "preparation": "Raw marinated mutton layered with Chinigura/Basmati rice and fried potatoes slow-cooked in a sealed pot with ghee and yogurt.",
        "ingredients": [
            {"name": "basmati_rice", "ratio": 0.48, "identification": "visible", "conf": 0.96},
            {"name": "mutton", "ratio": 0.30, "identification": "visible", "conf": 0.94},
            {"name": "potato", "ratio": 0.10, "identification": "visible", "conf": 0.91},
            {"name": "yogurt", "ratio": 0.04, "identification": "recipe_inferred", "conf": 0.86},
            {"name": "ghee", "ratio": 0.04, "identification": "recipe_inferred", "conf": 0.86},
            {"name": "onion", "ratio": 0.03, "identification": "recipe_inferred", "conf": 0.84},
            {"name": "mixed_spices", "ratio": 0.01, "identification": "recipe_inferred", "conf": 0.82},
        ],
    },
}

## 5. Dataset Builder & Stratified Train/Validation Split

Builds deterministic, stratified `train` and `val` JSONL files from `./food_dataset/train/<class>` using a shared prompt builder.

In [5]:
import json
import random
from PIL import Image

SYSTEM_PROMPT = (
    "You are the MealSense Bangladeshi culinary vision analyzer restricted strictly to 9 classes: "
    f"{', '.join(ALLOWED_FOOD_CLASSES)}. "
    "Given the image, DINOv3 prediction, and candidate ingredients, output strict JSON with visible "
    "ingredients and portion fullness. Never mark hidden oil or spices as visible."
)


def build_user_prompt(food_class: str, confidence: float, candidates: list[str]) -> str:
    return (
        f"{SYSTEM_PROMPT}\n"
        f"DINOv3 Class: {food_class} (confidence: {confidence:.2f})\n"
        f"Candidate Ingredients: {', '.join(candidates)}\n"
        "Extract visible ingredients and portion metadata as strict JSON."
    )


def build_stratified_jsonl_splits(cfg: MealSenseConfig) -> tuple[Path, Path]:
    valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
    rng = random.Random(cfg.seed)
    train_records: list[dict[str, Any]] = []
    val_records: list[dict[str, Any]] = []

    for food_class in ALLOWED_FOOD_CLASSES:
        class_dir = cfg.food_train_dir / food_class
        class_dir.mkdir(parents=True, exist_ok=True)
        kb = NINE_CLASS_KNOWLEDGE_BASE[food_class]

        image_files = [
            p for p in sorted(class_dir.iterdir())
            if p.is_file() and p.suffix.lower() in valid_exts
        ]
        if not image_files:
            fallback_img = class_dir / f"{food_class}_sample.jpg"
            Image.new("RGB", (cfg.image_size, cfg.image_size), color=(190, 150, 95)).save(
                fallback_img, format="JPEG"
            )
            image_files = [fallback_img]

        rng.shuffle(image_files)
        image_files = image_files[: cfg.max_images_per_class]

        val_count = max(1, int(len(image_files) * cfg.val_fraction)) if len(image_files) >= 5 else 0
        val_files = image_files[:val_count]
        train_files = image_files[val_count:]

        target_obj = VLMVisionOutput(
            dish_matches_prediction=True,
            visual_fullness_multiplier=1.0,
            portion_confidence="high",
            visible_ingredients=[
                VisibleIngredient(
                    name=ing["name"],
                    visual_confidence=ing["conf"],
                    estimated_volume_share=ing["ratio"],
                )
                for ing in kb["ingredients"]
                if ing["identification"] == "visible"
            ],
            observed_preparation_state=kb["preparation"],
            occluded_or_uncertain=False,
        )
        target_json = target_obj.model_dump_json()
        candidates = [ing["name"] for ing in kb["ingredients"]]

        for split_bucket, paths in ((train_records, train_files), (val_records, val_files)):
            for img_path in paths:
                split_bucket.append({
                    "image_path": str(img_path),
                    "dinov3_class": food_class,
                    "dinov3_confidence": 0.94,
                    "candidates": candidates,
                    "target_json": target_json,
                })

    rng.shuffle(train_records)
    rng.shuffle(val_records)

    train_path = cfg.artifacts_dir / "mealsense_9classes_train.jsonl"
    val_path = cfg.artifacts_dir / "mealsense_9classes_val.jsonl"
    for out_path, rows in ((train_path, train_records), (val_path, val_records)):
        with out_path.open("w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(
        f"Dataset splits ready -> Train: {len(train_records)} ({train_path.name}) | "
        f"Val: {len(val_records)} ({val_path.name})"
    )
    return train_path, val_path


TRAIN_JSONL_PATH, VAL_JSONL_PATH = build_stratified_jsonl_splits(CFG)

Dataset splits ready -> Train: 387 (mealsense_9classes_train.jsonl) | Val: 63 (mealsense_9classes_val.jsonl)


## 6. Model, Processor & LoRA Loader

Configures `Idefics3Processor` to avoid 17-crop upscaling, loads the base VLM with `dtype=` and `attn_implementation="sdpa"`, pins weights directly to GPU `cuda:0` when available, and applies LoRA via `TaskType.CAUSAL_LM`.

In [6]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig


def configure_processor(model_dir: Path, cfg: MealSenseConfig):
    proc = AutoProcessor.from_pretrained(str(model_dir), local_files_only=True)
    img_proc = getattr(proc, "image_processor", None)
    if img_proc is not None:
        if hasattr(img_proc, "do_image_splitting"):
            img_proc.do_image_splitting = cfg.enable_image_splitting
        if hasattr(img_proc, "size") and isinstance(img_proc.size, dict) and not cfg.enable_image_splitting:
            img_proc.size = {"longest_edge": cfg.image_size}
    return proc


def load_trainable_vlm(model_dir: Path, cfg: MealSenseConfig, runtime: RuntimeSpec):
    proc = configure_processor(model_dir, cfg)

    bnb_config = None
    if runtime.use_4bit and runtime.device == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=runtime.dtype,
        )

    load_kwargs: dict[str, Any] = {
        "dtype": runtime.dtype,
        "attn_implementation": "sdpa",
        "local_files_only": True,
    }
    if runtime.device == "cuda":
        load_kwargs["device_map"] = {"": 0}
    if bnb_config is not None:
        load_kwargs["quantization_config"] = bnb_config

    base_model = AutoModelForImageTextToText.from_pretrained(str(model_dir), **load_kwargs)
    if runtime.device == "cpu":
        base_model = base_model.to("cpu")

    if runtime.use_4bit:
        base_model = prepare_model_for_kbit_training(
            base_model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )
    else:
        base_model.config.use_cache = False

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return proc, model


processor, peft_model = load_trainable_vlm(LOCAL_MODEL_DIR, CFG, RUNTIME)

W1114 00:07:23.417000 26168 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1114 00:07:24.931000 26168 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
d:\CS\Hackathon\Mealsense_Models\.venv\Lib\site-packages\transformers\modeling_utils.py:5148: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)
Loading weights: 100%|██████████| 489/489 [00:00<00:00, 850.07it/s]


trainable params: 9,568,256 || all params: 517,050,560 || trainable%: 1.8505


In [9]:
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments


def load_rgb_image(image_path: str | Path, target_size: int) -> Image.Image:
    with Image.open(image_path) as img:        img = ImageOps.exif_transpose(img).convert("RGB")
        if img.size != (target_size, target_size):
            img = img.resize((target_size, target_size), Image.Resampling.BICUBIC)
        return img


class MealSenseVLMDataset(Dataset):
    def __init__(self, jsonl_file: Path, image_size: int) -> None:
        self.image_size = image_size
        with jsonl_file.open("r", encoding="utf-8") as f:
            self.items = [json.loads(line) for line in f if line.strip()]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        rec = self.items[idx]
        image = load_rgb_image(rec["image_path"], self.image_size)
        user_text = build_user_prompt(
            rec["dinov3_class"], rec["dinov3_confidence"], rec["candidates"]
        )
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": user_text},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": rec["target_json"]},
                ],
            },
        ]
        return {"messages": messages, "image": image}


class MultimodalSFTCollator:
    """Batches multimodal chat samples and masks everything before the assistant turn."""

    def __init__(self, proc) -> None:
        self.proc = proc
        self.image_token_id = getattr(proc, "image_token_id", None)
        if self.image_token_id is None:
            cand = proc.tokenizer.convert_tokens_to_ids("<image>")
            self.image_token_id = cand if isinstance(cand, int) and cand >= 0 else None

        probe = proc.apply_chat_template(
            [{"role": "user", "content": [{"type": "text", "text": "__PROBE__"}]}],
            tokenize=False,
            add_generation_prompt=True,
        )
        delimiter_str = probe.split("__PROBE__", 1)[1]
        self.delimiter_ids = proc.tokenizer.encode(delimiter_str, add_special_tokens=False)

    def _find_assistant_start(self, token_ids: list[int]) -> int:
        d_len = len(self.delimiter_ids)
        for i in range(len(token_ids) - d_len, -1, -1):
            if token_ids[i : i + d_len] == self.delimiter_ids:
                return i + d_len
        return 0

    def __call__(self, batch: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        texts = [
            self.proc.apply_chat_template(
                item["messages"], tokenize=False, add_generation_prompt=False
            )
            for item in batch
        ]
        images = [[item["image"]] for item in batch]

        model_inputs = self.proc(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = model_inputs["input_ids"].clone()
        for row_idx, seq in enumerate(labels.tolist()):
            assist_start = self._find_assistant_start(seq)
            labels[row_idx, :assist_start] = -100

        # Mask padding via attention_mask (safe even when pad_token_id == eos_token_id)
        if "attention_mask" in model_inputs:
            labels[model_inputs["attention_mask"] == 0] = -100
        if self.image_token_id is not None:
            labels[labels == self.image_token_id] = -100

        model_inputs["labels"] = labels
        return model_inputs


train_ds = MealSenseVLMDataset(TRAIN_JSONL_PATH, CFG.image_size)
val_ds = MealSenseVLMDataset(VAL_JSONL_PATH, CFG.image_size)
collator = MultimodalSFTCollator(processor)

is_cuda = RUNTIME.device == "cuda"
batch_size = (2 if RUNTIME.use_4bit else 4) if is_cuda else 1
grad_accum = (4 if RUNTIME.use_4bit else 2) if is_cuda else 4
optim_name = (
    "paged_adamw_8bit" if (is_cuda and RUNTIME.use_4bit)
    else ("adamw_torch_fused" if is_cuda else "adamw_torch")
)

training_args = TrainingArguments(
    output_dir=str(CFG.checkpoint_dir),
    num_train_epochs=CFG.num_train_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=CFG.learning_rate,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    logging_steps=5,
    eval_strategy="epoch" if len(val_ds) > 0 else "no",
    save_strategy="epoch",
    load_best_model_at_end=(len(val_ds) > 0),
    metric_for_best_model="eval_loss",
    save_total_limit=1,
    bf16=(RUNTIME.dtype == torch.bfloat16),
    fp16=(RUNTIME.dtype == torch.float16 and is_cuda),
    tf32=is_cuda,
    optim=optim_name,
    dataloader_pin_memory=is_cuda,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds if len(val_ds) > 0 else None,
    data_collator=collator,
)

trainer.train()

peft_model.save_pretrained(str(CFG.checkpoint_dir))
processor.save_pretrained(str(CFG.checkpoint_dir))
print(f"Best LoRA adapter & processor saved to: {CFG.checkpoint_dir}")

Epoch,Training Loss,Validation Loss
1,0.231495,0.136566
2,0.010496,0.007962
3,0.005096,0.005374


Best LoRA adapter & processor saved to: D:\CS\Hackathon\Mealsense_Models\ckpoint\mealsense-vlm-lora


## 8. Standalone Inference, Pydantic JSON Parsing & Nutrition Fusion

Supports running directly after training **or** standalone from a saved adapter in `CHECKPOINT_DIR`. Extracts JSON robustly, validates with `VLMVisionOutput`, and fuses visual portion and ingredient cues with FCTB 2013 nutrition data.

In [10]:
import re
from contextlib import nullcontext
from peft import PeftModel


def extract_json_object(raw_text: str) -> dict[str, Any]:
    cleaned = re.sub(r"```(?:json)?\s*|```", "", raw_text).strip()
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass
    return {}


def load_inference_model(cfg: MealSenseConfig, runtime: RuntimeSpec):
    proc_dir = cfg.checkpoint_dir if (cfg.checkpoint_dir / "preprocessor_config.json").exists() else cfg.local_base_model_dir
    proc = configure_processor(proc_dir, cfg)
    load_kwargs: dict[str, Any] = {
        "dtype": runtime.dtype,
        "attn_implementation": "sdpa",
        "local_files_only": True,
    }
    if runtime.device == "cuda":
        load_kwargs["device_map"] = {"": 0}

    base = AutoModelForImageTextToText.from_pretrained(str(cfg.local_base_model_dir), **load_kwargs)
    if runtime.device == "cpu":
        base = base.to("cpu")

    if (cfg.checkpoint_dir / "adapter_config.json").exists():
        model = PeftModel.from_pretrained(base, str(cfg.checkpoint_dir), local_files_only=True)
    else:
        model = base
    model.eval()
    return proc, model


def predict_mealsense_json(
    image_path: str | Path,
    dinov3_class: str,
    dinov3_confidence: float,
    model=None,
    proc=None,
    cfg: MealSenseConfig = CFG,
    runtime: RuntimeSpec = RUNTIME,
) -> dict[str, Any]:
    if dinov3_class not in ALLOWED_FOOD_CLASSES:
        raise ValueError(f"Unsupported food_class '{dinov3_class}'. Must be one of {ALLOWED_FOOD_CLASSES}")

    if model is None or proc is None:
        proc, model = load_inference_model(cfg, runtime)

    kb = NINE_CLASS_KNOWLEDGE_BASE[dinov3_class]
    candidates = [ing["name"] for ing in kb["ingredients"]]
    image = load_rgb_image(image_path, cfg.image_size)
    user_text = build_user_prompt(dinov3_class, dinov3_confidence, candidates)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": user_text},
        ],
    }]
    prompt = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[prompt], images=[[image]], padding=True, return_tensors="pt").to(runtime.device)

    amp_ctx = (
        torch.amp.autocast(device_type=runtime.device, dtype=runtime.dtype)
        if runtime.device == "cuda"
        else nullcontext()
    )
    with torch.inference_mode(), amp_ctx:
        out_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
        )

    trimmed = [o[len(i):] for i, o in zip(inputs["input_ids"], out_ids)]
    raw_text = proc.batch_decode(trimmed, skip_special_tokens=True)[0]

    parsed_dict = extract_json_object(raw_text)
    try:
        vlm_data = VLMVisionOutput.model_validate(parsed_dict)
    except Exception:
        vlm_data = VLMVisionOutput(
            visual_fullness_multiplier=1.0,
            portion_confidence="medium",
            observed_preparation_state=kb["preparation"],
        )

    visible_map = {v.name: v for v in vlm_data.visible_ingredients}
    est_serving_g = round(kb["standard_serving_g"] * vlm_data.visual_fullness_multiplier, 1)
    weight_range_g = (round(est_serving_g * 0.85, 1), round(est_serving_g * 1.15, 1))

    total_kcal = total_prot = total_carb = total_fat = 0.0
    ingredients_out: list[IngredientBreakdown] = []
    sources_set: set[str] = {kb["recipe_source"]}

    for ing in kb["ingredients"]:
        v_obs = visible_map.get(ing["name"])
        share = v_obs.estimated_volume_share if (v_obs and v_obs.estimated_volume_share > 0) else ing["ratio"]
        vis_conf = v_obs.visual_confidence if v_obs else ing["conf"]
        w_cooked = round(est_serving_g * share, 1)

        nutr = NUTRITION_DB.lookup(ing["name"])
        if nutr is not None:
            raw_equiv_g = w_cooked / max(0.1, nutr.cooking_yield_factor)
            factor = raw_equiv_g / 100.0
            total_kcal += factor * nutr.calories_kcal_100g
            total_prot += factor * nutr.protein_g_100g
            total_carb += factor * nutr.carbohydrates_g_100g
            total_fat += factor * nutr.fat_g_100g
            n_source = nutr.source_citation
            sources_set.add(n_source)
        else:
            n_source = "unmapped"

        ingredients_out.append(
            IngredientBreakdown(
                name=ing["name"],
                weight_g=w_cooked,
                identification=ing["identification"],
                confidence=round((vis_conf + dinov3_confidence) / 2.0, 2),
                nutrition_source=n_source,
                recipe_source=kb["recipe_source"],
            )
        )

    prediction = MealSensePrediction(
        food_class=dinov3_class,
        food_name=kb["food_name"],
        recognition_confidence=round(dinov3_confidence, 4),
        observed_preparation_state=vlm_data.observed_preparation_state or kb["preparation"],
        occluded_or_uncertain=vlm_data.occluded_or_uncertain,
        serving=ServingEstimate(
            estimated_weight_g=est_serving_g,
            weight_range_g=weight_range_g,
            confidence=vlm_data.portion_confidence,
        ),
        ingredients=ingredients_out,
        nutrition=MacroSummary(
            calories_kcal=round(total_kcal, 1),
            protein_g=round(total_prot, 1),
            carbohydrates_g=round(total_carb, 1),
            fat_g=round(total_fat, 1),
        ),
        sources=sorted(sources_set),
        requires_user_confirmation=(
            dinov3_confidence < 0.65
            or vlm_data.portion_confidence == "low"
            or vlm_data.occluded_or_uncertain
        ),
    )

    result_dict = prediction.model_dump()
    out_file = cfg.outputs_dir / f"{dinov3_class}_prediction.json"
    out_file.write_text(json.dumps(result_dict, indent=2), encoding="utf-8")
    return result_dict


sample_imgs = sorted((CFG.food_train_dir / "kacchi").glob("*.*"))
if sample_imgs:
    result_json = predict_mealsense_json(
        sample_imgs[0],
        dinov3_class="kacchi",
        dinov3_confidence=0.94,
        model=globals().get("peft_model"),
        proc=globals().get("processor"),
    )
    print(json.dumps(result_json, indent=2))
else:
    print("No sample images found in kacchi folder.")

{
  "food_class": "kacchi",
  "food_name": "Old Dhaka Mutton Kacchi Biryani",
  "recognition_confidence": 0.94,
  "observed_preparation_state": "Raw marinated mutton layered with Chinigura/Basmati rice and fried potatoes slow-cooked in a sealed pot with ghee and yogurt.",
  "occluded_or_uncertain": false,
  "serving": {
    "estimated_weight_g": 380.0,
    "weight_range_g": [
      323.0,
      437.0
    ],
    "confidence": "high"
  },
  "ingredients": [
    {
      "name": "basmati_rice",
      "weight_g": 182.4,
      "identification": "visible",
      "confidence": 0.95,
      "nutrition_source": "FCTB 2013 (Code 01_0012) - INFS, DU",
      "recipe_source": "https://www.fao.org/fileadmin/templates/food_composition/documents/FCT_10_2_14_final_version.pdf"
    },
    {
      "name": "mutton",
      "weight_g": 114.0,
      "identification": "visible",
      "confidence": 0.94,
      "nutrition_source": "FCTB 2013 (Code 10_0006) - INFS, DU",
      "recipe_source": "https://www.fao.org